# Stability Analysis: Verifying the Norm Bounds [HK Theorem 2.7]

This notebook provides a hands-on exploration of the **stability theory**
behind `stablefmmpy`, using the `StabilityAnalyzer`, `LeafMatrices`, and
`ScalingFactors` classes.

**Central result (HK Theorem 2.7):**
The balanced basis matrix $U$ satisfies $\|U\|_{\max} \le 1$ for all $r$ and all
point configurations in the LF regime — provided the scaling factors
$\lambda_{x,p} = \max\{1,\, p!\cdot(2/k\delta)^p\}$ are applied.

**References:**
- [HK] §2 Theorem 2.7 and Lemma 2.3 (`MichellePapers/HelmholtzKernel2D.pdf`)
- [M2D] §2 (`MichellePapers/Multipole2D.pdf`)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from stablefmmpy import (
    PointSet, ScalingFactors, HelmholtzKernel,
    LeafMatrices, StabilityAnalyzer,
)

print("stablefmmpy loaded.")
print(f"numpy {np.__version__}")


## Mathematical Foundation

### Scaling Factors [HK Eq. 2.17]

For a point set $X$ with bounding radius $\delta_X$, define:

$$\lambda_{x,p} = \max\left\{1,\; p! \cdot \left(\frac{2}{k\delta_X}\right)^p\right\}$$

These are computed on **log scale** via `scipy.special.gammaln` to prevent overflow:
$$\log\lambda_{x,p} = \max\left\{0,\; \log(p!) + p\log\frac{2}{k\delta_X}\right\}$$

### Norm Bound (Theorem 2.7)

The balanced basis entry satisfies:
$$U_{\text{bal}}[i,p] = J_{|p|}(k|z_i|) \cdot \lambda_{x,p} \cdot e^{ip\arg(z_i)}$$

where $z_i = k(x_i - o_X)$.  Since $|J_p(kr)| \le 1/\lambda_p$ in the LF regime,
this gives $|U_{\text{bal}}[i,p]| \le 1$ — **regardless of $r$**.


In [ ]:
# LF regime point sets: k=50, delta=0.005, k*delta=0.25 << r/e
rng  = np.random.default_rng(42)
k    = 50.0
N    = 40

X = PointSet.random_uniform(N, center=0+0j,    radius=0.005, rng=rng)
Y = PointSet.random_uniform(N, center=0.05+0j, radius=0.005, rng=rng)
q = rng.standard_normal(N) + 1j * rng.standard_normal(N)

print(f"X: center={X.center:.4g}, radius={X.radius:.4g}")
print(f"Y: center={Y.center:.4g}, radius={Y.radius:.4g}")
print(f"k*delta_X = {k * X.radius:.4g}  (LF regime if <= r/e for given r)")
print(f"Separation d = {abs(X.center - Y.center):.4g}")


## Section 1 — Norm Bound Verification

`StabilityAnalyzer.verify_norm_bounds()` computes the balanced and naive basis
matrices and returns a dictionary of key metrics.


In [ ]:
r  = 15
sa = StabilityAnalyzer(k=k, r=r)

bounds = sa.verify_norm_bounds(X, Y)
print("verify_norm_bounds() result:")
for key, val in bounds.items():
    print(f"  {key:<25} = {val}")


In [ ]:
assert bounds['theorem_satisfied'], "Theorem 2.7 VIOLATED!"
print()
print(f"Theorem 2.7 satisfied: ||U_balanced||_max = {bounds['U_max_balanced']:.6f} <= 1.0")
print(f"Naive comparison:       ||U_naive||_max    = {bounds['U_max_naive']:.6e}  (>> 1)")


## Section 2 — Rank Sweep: Stable vs. Naive Error

`StabilityAnalyzer.sweep_rank()` computes the balanced and naive relative errors
for a range of expansion orders $r$, using the same point sets and charge vector.


In [ ]:
r_values = [5, 8, 10, 12, 15, 18, 20]
rows     = sa.sweep_rank(X, Y, q, r_values=r_values)

print(f"{'r':>4}  {'Stable B':>10}  {'Stable err':>14}  {'Naive B':>16}  {'Naive err':>14}")
print("-" * 66)
for row in rows:
    Bb = f"{row['stable_B']:.4f}"
    eb = f"{row['stable_err']:.3e}" if np.isfinite(row['stable_err']) else "  Inf  "
    Bn = f"{row['naive_B']:.3e}"    if np.isfinite(row['naive_B'])    else "  Inf  "
    en = f"{row['naive_err']:.3e}"  if np.isfinite(row['naive_err'])  else "  Inf  "
    print(f"{row['r']:>4}  {Bb:>10}  {eb:>14}  {Bn:>16}  {en:>14}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

r_v  = [row['r']          for row in rows]
eb_v = [row['stable_err'] for row in rows]
en_v = [row['naive_err']  for row in rows]
Bb_v = [row['stable_B']   for row in rows]
Bn_v = [row['naive_B']    for row in rows]

# Left: relative error
ax = axes[0]
ax.semilogy(r_v, eb_v, 'b-o', lw=2, ms=7, label='Balanced FMM')
fin = [np.isfinite(e) for e in en_v]
r_fin = [r for r, m in zip(r_v, fin) if m]
e_fin = [e for e, m in zip(en_v, fin) if m]
r_inf = [r for r, m in zip(r_v, fin) if not m]
if r_fin:
    ax.semilogy(r_fin, e_fin, 'r-s', lw=2, ms=7, label='Naive FMM')
if r_inf:
    y_top = ax.get_ylim()[1] * 10
    ax.plot(r_inf, [y_top] * len(r_inf), 'rv', ms=14, label='Naive = Inf')
ax.axhline(1e-15, ls='--', color='gray', alpha=0.6, label='Machine precision')
ax.set_xlabel('Expansion order $r$', fontsize=12)
ax.set_ylabel('Relative error', fontsize=12)
ax.set_title('Error vs. $r$', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: B-factor
ax = axes[1]
ax.semilogy(r_v, Bb_v, 'b-o', lw=2, ms=7, label='$\|B\|_{\max}$ balanced')
fin_B = [np.isfinite(b) for b in Bn_v]
r_finB = [r for r, m in zip(r_v, fin_B) if m]
B_finB = [b for b, m in zip(Bn_v, fin_B) if m]
if r_finB:
    ax.semilogy(r_finB, B_finB, 'r-s', lw=2, ms=7, label='$\|B\|_{\max}$ naive')
ax.set_xlabel('Expansion order $r$', fontsize=12)
ax.set_ylabel('Balance factor $\|B\|_{\max}$', fontsize=12)
ax.set_title('Balance Factor vs. $r$', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

fig.suptitle(f'Rank Sweep (k={k}, N={N}, LF regime)', fontsize=13, y=1.01)
fig.tight_layout()
plt.savefig('03_sweep_rank.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 03_sweep_rank.png")


## Section 3 — Overflow Threshold

`StabilityAnalyzer.find_overflow_threshold()` scans increasing $r$ values and
returns the first $r$ at which the **naive** $\|B\|_{\max}$ overflows to `Inf`.
This threshold depends on $k$ and $\delta$ (the point-set geometry).


In [ ]:
threshold = sa.find_overflow_threshold(X, Y, r_max=200, step=5)
print(f"Overflow threshold: naive B first overflows at r = {threshold}")
print()
# Verify balanced stays finite at the threshold
lm     = LeafMatrices(r=threshold, k=k)
sf_x   = ScalingFactors(threshold, k, X.radius)
sf_y   = ScalingFactors(threshold, k, Y.radius)
B_bal  = lm.build_B_lf(X, Y, sf_x, sf_y, balanced=True)
B_nai  = lm.build_B_lf(X, Y, sf_x, sf_y, balanced=False)
print(f"At r={threshold}: balanced ||B||_max = {lm.balance_factor(B_bal):.4f}")
print(f"At r={threshold}: naive    ||B||_max = {lm.balance_factor(B_nai)}")


## Section 4 — LeafMatrices: LF vs. HF Regime

`LeafMatrices.regime(delta)` decides whether to use the low-frequency (Bessel/Hankel)
or high-frequency (DFT) basis for a given bounding radius $\delta$:

$$\text{LF if } k\delta \le r/e, \quad \text{HF if } k\delta > r/e$$

The LF basis uses Miller backward recurrence; the HF basis uses equispaced plane
waves — both are numerically stable, but for different reasons.


In [ ]:
lm = LeafMatrices(r=20, k=50.0)

delta_values = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0]
threshold_delta = lm.r / (lm.k * np.e)
print(f"LF/HF boundary: k*delta = r/e -> delta* = {threshold_delta:.5f}")
print()
print(f"{'delta':>10}  {'k*delta':>10}  {'regime':>8}")
print("-" * 34)
for d in delta_values:
    reg = lm.regime(d)
    print(f"{d:>10.4g}  {lm.k * d:>10.4f}  {reg:>8}")


In [ ]:
# Build balanced vs naive U for the LF configuration
sf_bal = ScalingFactors(r=20, k=k, delta=X.radius)

# Balanced basis (should have ||U||_max <= 1)
U_bal  = lm.build_basis_lf(X, sf_bal, balanced=True)

# Naive basis (no scaling)
sf_nai      = ScalingFactors.__new__(ScalingFactors)
sf_nai.r    = 20
sf_nai.k    = k
sf_nai.delta = X.radius
sf_nai._log_lam = np.zeros(21)
U_nai  = lm.build_basis_lf(X, sf_nai, balanced=False)

print(f"Balanced U:  shape={U_bal.shape},  ||U||_max = {np.max(np.abs(U_bal)):.6f}  (<= 1)")
print(f"Naive    U:  shape={U_nai.shape},  ||U||_max = {np.max(np.abs(U_nai)):.3e}  (>> 1)")
print()

# Histogram of |U[i,p]| across all entries
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.abs(U_bal).ravel(), bins=40, alpha=0.7, color='steelblue',
        label='Balanced $|U_{bal}|$')
ax.axvline(1.0, color='red', lw=2, ls='--', label='Bound: $\|U\|_{\max} = 1$')
ax.set_xlabel('Entry magnitude $|U[i,p]|$', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Balanced Basis Entry Distribution (LF regime)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig('03_basis_norm.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 03_basis_norm.png")


## Section 5 — ScalingFactors: The $\lambda_{x,p}$ Values

`ScalingFactors` precomputes $\log\lambda_{x,p}$ via `gammaln` to avoid overflow.
The `as_array()` method returns $\lambda_p$ in linear scale (may be astronomically large);
`log_array()` always returns finite values.


In [ ]:
sf = ScalingFactors(r=20, k=k, delta=X.radius)

print(f"ScalingFactors(r=20, k={k}, delta={X.radius:.4f})")
print()
print(f"  k*delta = {k * X.radius:.4f}  (LF regime)")
print()
print(f"{'p':>4}  {'log(lambda_p)':>16}  {'lambda_p':>20}")
print("-" * 46)
for p in range(min(8, sf.r + 1)):
    ll = sf.log_lam(p)
    lv = sf[p]
    lv_str = f"{lv:.4e}" if lv < 1e100 else "  >> 1e100 (large)"
    print(f"{p:>4}  {ll:>16.6f}  {lv_str:>20}")

print()
print("Note: lambda_0 = 1 (p=0 term is never scaled).")
print("For large p, lambda_p >> 1 prevents J_p(kz) from underflowing to 0 in U.")


## Conclusions

| Quantity | Balanced | Naive |
|----------|---------|-------|
| $\|U\|_{\max}$ | $\le 1$ (Theorem 2.7) | $\gg 1$ (overflow) |
| $\|B\|_{\max}$ | $O(1)$, constant in $r$ | $\to \infty$ at threshold $r$ |
| Relative error | $\approx 10^{-15}$ | `NaN` / `Inf` |

**Key takeaways:**
1. `ScalingFactors` computes $\lambda_{x,p}$ entirely in log-space via `gammaln`,
   so even astronomically large values are handled without overflow.
2. The balanced $\|B\|_{\max}$ is constant with respect to $r$ — it depends only
   on $kd$ (the centre-to-centre distance) and is bounded by $|H_0(kd)|$ [HK Lemma 2.3].
3. `StabilityAnalyzer.find_overflow_threshold()` can be used as a diagnostic tool
   to determine the safe operating range of the naive method for any geometry.
4. The LF/HF regime boundary (`LeafMatrices.regime()`) is the key decision point
   of the wideband algorithm: crossing it triggers a switch from Bessel to DFT basis.
